<div style="background:#ffffff;border:1px solid #000000;border-radius:0.75rem;padding:24px 28px;color:#111827;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;">
  <div style="font-size:0.85em;letter-spacing:0.12em;text-transform:uppercase;color:#6e7781;">Stanford Musculoskeletal Imaging Workshop · November 2026</div>
  <h1 style="margin:6px 0 4px;font-size:2.2em;color:#111827;font-weight:700;"><span style="color:#f97316;">MuscleMap</span> hands-on</h1>
  <div style="font-size:1.15em;color:#5c5962;">Segmenting a lumbar spine MRI, from image to muscle size and fat</div>
  <img src="../assets/v1.4_figure.gif" alt="MuscleMap whole-body muscle segmentation" style="display:block;width:100%;border-radius:0.5rem;margin:18px 0 4px;">
  <table style="margin-top:20px;border-collapse:collapse;width:100%;color:inherit;font-size:0.95em;background:transparent;">
    <tr style="background:transparent;color:#6e7781;text-align:left;"><th style="padding:6px 8px;border-bottom:2px solid #f97316;">Step</th><th style="padding:6px 8px;border-bottom:2px solid #f97316;">What happens</th><th style="padding:6px 8px;border-bottom:2px solid #f97316;">Time</th></tr>
    <tr style="background:transparent;"><td style="padding:6px 8px;color:inherit;">1 · Load</td><td style="padding:6px 8px;color:inherit;">Open the example MRI and view the four slices</td><td style="padding:6px 8px;color:inherit;">1 min</td></tr>
    <tr style="background:transparent;"><td style="padding:6px 8px;color:inherit;">2 · Segment</td><td style="padding:6px 8px;color:inherit;">Run <code>mm_segment</code> and read its log</td><td style="padding:6px 8px;color:inherit;">2–10 min</td></tr>
    <tr style="background:transparent;"><td style="padding:6px 8px;color:inherit;">3 · Inspect</td><td style="padding:6px 8px;color:inherit;">Interactive overlay viewer</td><td style="padding:6px 8px;color:inherit;">2 min</td></tr>
    <tr style="background:transparent;"><td style="padding:6px 8px;color:inherit;">4 · Measure</td><td style="padding:6px 8px;color:inherit;">Cross-sectional area of psoas, multifidus and erector spinae</td><td style="padding:6px 8px;color:inherit;">1 min</td></tr>
    <tr style="background:transparent;"><td style="padding:6px 8px;color:inherit;">5 · Quantify fat</td><td style="padding:6px 8px;color:inherit;">Run <code>mm_extract_metrics</code> with GMM and k-means</td><td style="padding:6px 8px;color:inherit;">2–5 min</td></tr>
    <tr style="background:transparent;"><td style="padding:6px 8px;color:inherit;">6 · Fat maps</td><td style="padding:6px 8px;color:inherit;">Fat overlay and fat percentage per muscle</td><td style="padding:6px 8px;color:inherit;">2 min</td></tr>
  </table>
</div>

<div style="border:1px solid #000000;border-left:5px solid #f97316;background:#ffffff;border-radius:0.5rem;padding:10px 16px;margin:10px 0;color:#111827;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;"><b>Kernel.</b> The top-right corner of this notebook must say <b>Python (MuscleMap)</b>. If it does not, choose it under <i>Kernel → Change Kernel</i>. The setup instructions are in the README of this repository.</div>

<div style="border:1px solid #000000;border-left:5px solid #f97316;background:#ffffff;border-radius:0.5rem;padding:10px 16px;margin:10px 0;color:#111827;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;"><b>First run.</b> The first time <code>mm_segment</code> runs it downloads the model weights (about 100 MB). Do this at home, not on conference Wi-Fi.</div>

Run the cells one by one with **Shift + Enter**.

## Setup

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import nibabel as nib
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, HTML

pio.templates.default = "plotly_white"
CLEAR = "rgba(0,0,0,0)"                                 # transparent figure background
SLIDER = dict(font=dict(color="#333"), bgcolor="#eeeeee", bordercolor="#dddddd", tickcolor="#999999")

# Locate the repository root (the folder that contains data/MuscleMap), starting from this notebook
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "MuscleMap").is_dir())
image_path = repo_root / "data" / "MuscleMap" / "sub-01_T2w_L4-L5.nii.gz"

# mm_segment lives next to the kernel's Python: <env>/Scripts on Windows, <env>/bin on macOS and Linux
env_dir = Path(sys.executable).parent
candidates = [env_dir / "Scripts" / "mm_segment.exe", env_dir / "mm_segment", shutil.which("mm_segment")]
mm_segment = next((str(c) for c in candidates if c and Path(c).exists()), None)

def status_table(rows):
    """rows: list of (label, ok, text)"""
    dot = lambda ok: f'<span style="color:{"#3ddc84" if ok else "#ff5c5c"};font-size:1.1em;">●</span>'
    html = '<table style="border-collapse:collapse;font-family:ui-monospace,Menlo,Consolas,monospace;font-size:0.9em;">'
    for label, ok, text in rows:
        html += f'<tr><td style="padding:4px 10px 4px 0;color:#8a8a93;">{label}</td><td style="padding:4px 8px;">{dot(ok)}</td><td style="padding:4px 0;">{text}</td></tr>'
    display(HTML(html + "</table>"))

status_table([
    ("Kernel",     "MuscleMap" in sys.executable, sys.executable),
    ("mm_segment", mm_segment is not None,        mm_segment or "not found: is MuscleMap installed in this environment?"),
    ("Image",      image_path.exists(),           str(image_path)),
])

Every structure gets one colour that is used in the table, the overlay and the 3D view. Left and right of the same muscle share a hue; the right side is the lighter shade.

In [ ]:
LABELS = {
    1101: "Left Levator Scapulae",
    1102: "Right Levator Scapulae",
    1111: "Left Semispinalis Cervicis and Multifidus",
    1112: "Right Semispinalis Cervicis and Multifidus",
    1121: "Left Semispinalis Capitis",
    1122: "Right Semispinalis Capitis",
    1131: "Left Splenius Capitis",
    1132: "Right Splenius Capitis",
    1141: "Left Sternocleidomastoid",
    1142: "Right Sternocleidomastoid",
    1151: "Left Longus Colli",
    1152: "Right Longus Colli",
    1161: "Left Trapezius",
    1162: "Right Trapezius",
    2101: "Left Supraspinatus",
    2102: "Right Supraspinatus",
    2111: "Left Subscapularis",
    2112: "Right Subscapularis",
    2121: "Left Infraspinatus",
    2122: "Right Infraspinatus",
    2141: "Left Deltoid",
    2142: "Right Deltoid",
    4101: "Left Rhomboid",
    4102: "Right Rhomboid",
    5101: "Left Thoracolumbar Multifidus",
    5102: "Right Thoracolumbar Multifidus",
    5111: "Left Erector Spinae",
    5112: "Right Erector Spinae",
    5121: "Left Psoas Major",
    5122: "Right Psoas Major",
    5131: "Left Quadratus Lumborum",
    5132: "Right Quadratus Lumborum",
    5141: "Left Latissimus Dorsi",
    5142: "Right Latissimus Dorsi",
    6101: "Left Gluteus Minimus",
    6102: "Right Gluteus Minimus",
    6111: "Left Gluteus Medius",
    6112: "Right Gluteus Medius",
    6121: "Left Gluteus Maximus",
    6122: "Right Gluteus Maximus",
    6131: "Left Tensor Fasciae Latae",
    6132: "Right Tensor Fasciae Latae",
    6141: "Left Iliacus",
    6142: "Right Iliacus",
    6151: "Left Ilium",
    6152: "Right Ilium",
    6160: "Sacrum",
    6171: "Left Femur",
    6172: "Right Femur",
    6181: "Left Piriformis",
    6182: "Right Piriformis",
    6191: "Left Pectineus",
    6192: "Right Pectineus",
    6201: "Left Obturator Internus",
    6202: "Right Obturator Internus",
    6211: "Left Obturator Externus",
    6212: "Right Obturator Externus",
    6221: "Left Gemelli and Quadratus Femoris",
    6222: "Right Gemelli and Quadratus Femoris",
    7101: "Left Vastus Lateralis",
    7102: "Right Vastus Lateralis",
    7111: "Left Vastus Intermedius",
    7112: "Right Vastus Intermedius",
    7121: "Left Vastus Medialis",
    7122: "Right Vastus Medialis",
    7131: "Left Rectus Femoris",
    7132: "Right Rectus Femoris",
    7141: "Left Sartorius",
    7142: "Right Sartorius",
    7151: "Left Gracilis",
    7152: "Right Gracilis",
    7161: "Left Semimembranosus",
    7162: "Right Semimembranosus",
    7171: "Left Semitendinosus",
    7172: "Right Semitendinosus",
    7181: "Left Biceps Femoris Long Head",
    7182: "Right Biceps Femoris Long Head",
    7191: "Left Biceps Femoris Short Head",
    7192: "Right Biceps Femoris Short Head",
    7201: "Left Adductor Magnus",
    7202: "Right Adductor Magnus",
    7211: "Left Adductor Longus",
    7212: "Right Adductor Longus",
    7221: "Left Adductor Brevis",
    7222: "Right Adductor Brevis",
    7231: "Left Patella",
    7232: "Right Patella",
    8101: "Left Tibialis Anterior",
    8102: "Right Tibialis Anterior",
    8111: "Left Tibialis Posterior",
    8112: "Right Tibialis Posterior",
    8121: "Left Peroneus Longus",
    8122: "Right Peroneus Longus",
    8131: "Left Soleus",
    8132: "Right Soleus",
    8141: "Left Medial Gastrocnemius",
    8142: "Right Medial Gastrocnemius",
    8151: "Left Lateral Gastrocnemius",
    8152: "Right Lateral Gastrocnemius",
    8161: "Left Tibia",
    8162: "Right Tibia",
    8171: "Left Fibula",
    8172: "Right Fibula",
    8181: "Left Flexor Hallucis Longus",
    8182: "Right Flexor Hallucis Longus",
    8191: "Left Extensor Digitorum Longus",
    8192: "Right Extensor Digitorum Longus",
    8201: "Left Flexor Digitorum Longus",
    8202: "Right Flexor Digitorum Longus",
    8211: "Left Popliteus",
    8212: "Right Popliteus",
    8221: "Left Plantaris",
    8222: "Right Plantaris",
}

PALETTE = ["#4F8EF7", "#F28E2B", "#E15759", "#59A14F", "#B07AA1", "#EDC948",
           "#76B7B2", "#FF9DA7", "#9C755F", "#17BECF", "#D4A017", "#8CD17D",
           "#F1CE63", "#A0CBE8", "#FFBE7D", "#D37295", "#B6992D", "#499894"]

def _lighten(hex_color, amount=0.38):
    r, g, b = (int(hex_color[i:i + 2], 16) for i in (1, 3, 5))
    r, g, b = (int(c + (255 - c) * amount) for c in (r, g, b))
    return f"#{r:02X}{g:02X}{b:02X}"

def color_for(label, _cache={}):
    """Same hue for left (…1) and right (…2) of a muscle; unpaired labels get their own colour."""
    muscle = label // 10
    if muscle not in _cache:
        _cache[muscle] = PALETTE[len(_cache) % len(PALETTE)]
    base = _cache[muscle]
    return _lighten(base) if label % 10 == 2 else base

## 1 · Load and view the image

The example is an axial T2-weighted MRI of the lumbar spine at the L4–L5 disc: four slices of 4.4 mm, 0.69 mm in-plane, shown side by side. The image is shown in radiological convention: the patient's right is on the left of the image, anterior is at the top.

In [ ]:
img = nib.load(image_path)
data = img.get_fdata()
voxel = tuple(float(v) for v in img.header.get_zooms())
n_slices = data.shape[2]

display(pd.DataFrame({
    "value": [str(data.shape), " × ".join(f"{v:.2f}" for v in voxel) + " mm",
              "".join(nib.aff2axcodes(img.affine)), f"{np.prod(voxel) / 1000:.3f} ml"],
}, index=["shape (x, y, z)", "voxel size", "orientation", "voxel volume"]).style.set_properties(**{"text-align": "left"}))

# Grey-level window for display, shared by all views
lo, hi = np.percentile(data, [1, 99.5])

def to_display(a):
    """Reorient a (x, y) slice for display: anterior at the top, patient's right on the left."""
    return np.flipud(a.T)

# Crop to the body (plus a margin) so the panels spend their pixels on anatomy rather than air.
# Rows and columns whose mean intensity is below 10 % of the window maximum are air.
mean_slice = to_display(data.mean(axis=2))
rows_ = np.where(mean_slice.mean(axis=1) > 0.10 * hi)[0]
cols_ = np.where(mean_slice.mean(axis=0) > 0.10 * hi)[0]
pad = 8
crop = (slice(max(rows_.min() - pad, 0), rows_.max() + pad), slice(max(cols_.min() - pad, 0), cols_.max() + pad))

crop_rows = crop[0].stop - crop[0].start
crop_cols = crop[1].stop - crop[1].start
def panel_height(n_cols, fig_width=1000, spacing=12):
    """Pixel height of one image panel when n_cols cropped slices share the figure width."""
    return (fig_width - spacing * (n_cols - 1)) / n_cols * crop_rows / crop_cols

def to_rgb(k):
    """Slice k as a cropped RGB uint8 array."""
    g = np.clip((to_display(data[:, :, k])[crop] - lo) / (hi - lo), 0, 1)
    g = (g * 255).astype(np.uint8)
    return np.stack([g, g, g], axis=-1)

def slice_row(frames, title, extra_sliders=()):
    """One panel per axial slice, side by side. frames: list (per slice) of trace lists."""
    fig = make_subplots(rows=1, cols=len(frames), horizontal_spacing=0.01,
                        subplot_titles=[f"slice {k}" for k in range(len(frames))])
    for k, traces in enumerate(frames):
        for t in traces:
            fig.add_trace(t, row=1, col=k + 1)
        fig.update_xaxes(visible=False, row=1, col=k + 1)
        fig.update_yaxes(visible=False, scaleanchor=f"x{k + 1 if k else ''}", row=1, col=k + 1)
    top = 60 if title else 36
    fig.update_layout(title=title, height=int(top + panel_height(len(frames)) + 20 + 80 * len(extra_sliders)),
                      margin=dict(l=10, r=10, t=top, b=10 + 80 * len(extra_sliders)),
                      paper_bgcolor=CLEAR, plot_bgcolor=CLEAR, sliders=list(extra_sliders))
    return fig

slice_row([[go.Image(z=to_rgb(k), hoverinfo="skip")] for k in range(n_slices)], None).show()

## 2 · Run MuscleMap

`mm_segment` takes the image (`-i`), the model (`-r`) and an output folder (`-o`). The whole-body model segments 113 muscles and bones and is contrast-agnostic. The output is `sub-01_T2w_L4-L5_dseg.nii.gz` plus a JSON sidecar with the model version and settings.

<div style="border:1px solid #000000;border-left:5px solid #f97316;background:#ffffff;border-radius:0.5rem;padding:10px 16px;margin:10px 0;color:#111827;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;">MuscleMap uses the GPU when it finds one and the CPU otherwise. On a CPU this image takes a few minutes. The MuscleMap log streams into the notebook while it runs; the summary underneath shows which device was used.</div>

In [ ]:
output_dir = image_path.parent                       # write the segmentation next to the input image
command = [mm_segment, "-i", str(image_path), "-r", "wholebody", "-o", str(output_dir)]
display(HTML('<div style="font-family:ui-monospace,Menlo,Consolas,monospace;background:#17171a;border-radius:8px;padding:10px 14px;color:#e6e6e6;">'
             '<span style="color:#8a8a93;">$</span> mm_segment ' + " ".join(command[1:]) + "</div>"))

env = {**os.environ, "MPLBACKEND": "Agg"}             # Jupyter's inline plotting backend must not leak into mm_segment

def log_box(lines, title):
    """The MuscleMap log as an expanded, scrollable block."""
    body = "\n".join(lines).replace("<", "&lt;") or "..."
    return HTML(f'<details open style="margin-top:8px;"><summary style="cursor:pointer;color:#6e7781;">{title}</summary>'
                f'<pre style="background:#0f0f10;color:#cfcfd4;padding:12px;border-radius:8px;font-size:0.8em;max-height:420px;overflow:auto;">{body}</pre></details>')

log_lines, t0 = [], time.time()
live = display(log_box([], "MuscleMap log - starting"), display_id=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in process.stdout:                          # stream the log into the notebook while MuscleMap runs
    log_lines.append(line.rstrip("\n"))
    live.update(log_box(log_lines, f"MuscleMap log - running {time.time() - t0:.0f} s"))
process.wait()
duration = time.time() - t0

seg_path = output_dir / image_path.name.replace(".nii.gz", "_dseg.nii.gz")
json_path = seg_path.with_name(seg_path.name.replace(".nii.gz", ".json"))
info = json.loads(json_path.read_text()) if json_path.exists() else {}
ok = process.returncode == 0 and seg_path.exists()

summary = [("Result", ok, "segmentation written" if ok else f"failed (exit code {process.returncode})"),
           ("Duration", True, f"{duration:.0f} s"),
           ("Device", True, str(info.get("device", "?")).upper()),
           ("Model", True, f'{info.get("model", "?")}  v{info.get("model_version", "?")}'),
           ("Output", seg_path.exists(), str(seg_path))]
live.update(log_box(log_lines, f"MuscleMap log - {len(log_lines)} lines - {duration:.0f} s"))
status_table(summary)

## 3 · Overlay viewer

The whole-body model labels 14 structures in this image. From here on we work with three muscles only: **psoas major**, **thoracolumbar multifidus** and **erector spinae**, left and right. The cell below keeps those six labels, saves them as a second segmentation file (`…_dseg-paraspinal.nii.gz`) that the later steps use, and shows them on top of the image. Change the transparency with the slider and hover over a muscle to see its name.

In [ ]:
seg_img = nib.load(seg_path)
seg_all = seg_img.get_fdata().astype(int)
assert seg_all.shape == data.shape, "segmentation and image have different shapes"

TARGETS = ["Psoas Major", "Thoracolumbar Multifidus", "Erector Spinae"]
def structure_of(label):
    return LABELS.get(label, str(label)).replace("Left ", "").replace("Right ", "")

all_labels = [int(v) for v in np.unique(seg_all) if v != 0]
present = [v for v in all_labels if structure_of(v) in TARGETS]
seg = np.where(np.isin(seg_all, present), seg_all, 0)          # the three muscles only
colors = {v: color_for(v) for v in present}

seg_sel_path = output_dir / image_path.name.replace(".nii.gz", "_dseg-paraspinal.nii.gz")
nib.save(nib.Nifti1Image(seg.astype(np.int16), seg_img.affine, seg_img.header), seg_sel_path)
print(f"{len(all_labels)} structures segmented, {len(present)} kept:", ", ".join(LABELS[v] for v in present))
print("saved", seg_sel_path.name)

def overlay_rgba(k, label_names):
    """Segmentation of slice k as an RGBA uint8 array (transparent where there is no label)."""
    s = to_display(seg[:, :, k])[crop]
    rgba = np.zeros(s.shape + (4,), dtype=np.uint8)
    names = np.full(s.shape, "", dtype=object)
    for v, c in colors.items():
        m = s == v
        rgba[m] = [int(c[i:i + 2], 16) for i in (1, 3, 5)] + [255]
        names[m] = label_names[v]
    return rgba, names

frames = []
for k in range(n_slices):
    rgba, names = overlay_rgba(k, LABELS)
    frames.append([go.Image(z=to_rgb(k), hoverinfo="skip"),
                   go.Image(z=rgba, colormodel="rgba", opacity=0.5, text=names,
                            hovertemplate="%{text}<extra></extra>")])

opacity_steps = [dict(method="restyle", label=f"{a:.1f}",
                      args=[{"opacity": a}, [2 * k + 1 for k in range(n_slices)]]) for a in np.arange(0, 1.01, 0.1)]
opacity_slider = dict(active=5, steps=opacity_steps, currentvalue=dict(prefix="overlay opacity "), pad=dict(t=10),
                      **SLIDER)

slice_row(frames, None, [opacity_slider]).show()

## 4 · Volume and cross-sectional area

For every slice the cross-sectional area (CSA) is the number of labelled pixels times the pixel area. From that follow three measures per muscle and side:

* **CSA (mm²)**: the total cross-sectional area, summed over all slices.
* **aCSA (mm²)**: the average cross-sectional area per slice, CSA divided by the number of slices.
* **Volume (ml)**: CSA times the slice thickness.

In [ ]:
pixel_mm2 = voxel[0] * voxel[1]                     # in-plane pixel area
slice_mm = voxel[2]                                  # slice thickness

# Order the slices from cranial to caudal using the scanner coordinate of each slice centre
z_mm = np.array([(img.affine @ [data.shape[0] / 2, data.shape[1] / 2, k, 1])[2] for k in range(n_slices)])
order = np.argsort(-z_mm)                            # highest (most cranial) first

rows = []
for v in present:
    name = LABELS.get(v, str(v))
    structure = structure_of(v)
    csa = np.array([(seg[:, :, k] == v).sum() * pixel_mm2 for k in range(n_slices)])
    rows.append(dict(structure=structure, side="Left" if name.startswith("Left ") else "Right", label=v,
                     **{"volume (ml)": csa.sum() * slice_mm / 1000, "CSA (mm²)": csa.sum(), "aCSA (mm²)": csa.mean()},
                     csa_cranial_to_caudal=csa[order]))
muscles = pd.DataFrame(rows).set_index(["structure", "side"]).reindex(TARGETS, level=0)

table = muscles[["volume (ml)", "CSA (mm²)", "aCSA (mm²)"]]
styled = (table.style
          .format({"volume (ml)": "{:.1f}", "CSA (mm²)": "{:.0f}", "aCSA (mm²)": "{:.0f}"})
          .apply(lambda col: [f"border-left: 10px solid {colors[muscles.loc[i, 'label']]}" for i in col.index], subset=["volume (ml)"])
          .set_caption(f"{n_slices} slices of {slice_mm:.1f} mm at L4–L5 · CSA summed over the slices · aCSA per slice")
          .set_table_styles([{"selector": "caption", "props": "caption-side: top; color: #6e7781; padding-bottom: 6px;"}]))
display(styled)

### CSA along the muscle, normalised to 0–100 %

As in [Stewart et al. 2024](https://pmc.ncbi.nlm.nih.gov/articles/PMC11634488/), the per-slice values are expressed along the length of the imaged muscle: the most cranial slice is 0 %, the most caudal slice 100 %, and the values in between are interpolated with a spline to 101 points. One panel per muscle; solid is left, dashed is right, the dots are the measured slices.

In [ ]:
try:
    from scipy.interpolate import CubicSpline
    def resample(y, x_new):
        return CubicSpline(np.linspace(0, 100, len(y)), y)(x_new)
except ImportError:                                  # fall back to linear interpolation
    def resample(y, x_new):
        return np.interp(x_new, np.linspace(0, 100, len(y)), y)

length_pct = np.linspace(0, 100, 101)
x_slices = np.linspace(0, 100, n_slices)

fig = make_subplots(rows=1, cols=len(TARGETS), subplot_titles=TARGETS, shared_yaxes=True, horizontal_spacing=0.04)
for col, structure in enumerate(TARGETS, start=1):
    for side, dash in (("Left", "solid"), ("Right", "dash")):
        if (structure, side) not in muscles.index:
            continue
        y = muscles.loc[(structure, side), "csa_cranial_to_caudal"]
        c = colors[muscles.loc[(structure, side), "label"]]
        fig.add_trace(go.Scatter(x=length_pct, y=resample(y, length_pct), mode="lines", name=side, legendgroup=side,
                                 showlegend=(col == 1), line=dict(color=c, width=3, dash=dash),
                                 hovertemplate="%{x:.0f} %<br>%{y:.0f} mm²<extra>" + f"{side} {structure}</extra>"),
                      row=1, col=col)
        fig.add_trace(go.Scatter(x=x_slices, y=y, mode="markers", showlegend=False, legendgroup=side,
                                 marker=dict(color=c, size=9, line=dict(color="white", width=1)), hoverinfo="skip"),
                      row=1, col=col)
    fig.update_xaxes(title_text="Length (%)", range=[0, 100], dtick=20, gridcolor="#e5e7eb", row=1, col=col)
fig.update_yaxes(title_text="CSA (mm²)", rangemode="tozero", gridcolor="#e5e7eb", row=1, col=1)
fig.update_layout(height=430, paper_bgcolor=CLEAR, plot_bgcolor=CLEAR, margin=dict(l=10, r=10, t=60, b=10),
                  legend=dict(bgcolor="rgba(0,0,0,0)", orientation="h", y=-0.22, x=0.5, xanchor="center"))
fig.show()

## 5 · Fat infiltration with mm_extract_metrics

`mm_extract_metrics` quantifies each segmented muscle. On a T1- or T2-weighted image it clusters the voxel intensities inside every muscle into **three components**: muscle (dark), undefined (intermediate) and fat (bright). Two clustering methods are available and we run both: **Gaussian mixture modelling** (`gmm`) and **k-means** (`kmeans`). Each run writes a CSV with the results and NIfTI maps of the components; the fat map is what we look at next. We pass the filtered segmentation from step 3, so only the six paraspinal labels are analysed.

<div style="border:1px solid #000000;border-left:5px solid #f97316;background:#ffffff;border-radius:0.5rem;padding:10px 16px;margin:10px 0;color:#111827;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;">The clustering runs separately for each label with 20 restarts, so expect a minute or two per method. Use <code>--qc</code> on the command line to inspect and adjust the thresholds interactively.</div>

In [ ]:
metrics_runs = {}
for method in ("gmm", "kmeans"):
    cmd = [str(Path(mm_segment).with_name(Path(mm_segment).name.replace("mm_segment", "mm_extract_metrics"))),
           "-m", method, "-r", "wholebody", "-i", str(image_path), "-s", str(seg_sel_path), "-c", "3", "-o", str(output_dir)]
    display(HTML('<div style="font-family:ui-monospace,Menlo,Consolas,monospace;background:#17171a;border-radius:8px;padding:10px 14px;color:#e6e6e6;margin-top:14px;">'
                 '<span style="color:#8a8a93;">$</span> mm_extract_metrics ' + " ".join(cmd[1:]) + "</div>"))
    lines, t0 = [], time.time()
    live = display(log_box([], f"MuscleMap log ({method}) - starting"), display_id=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in proc.stdout:
        lines.append(line.rstrip("\n"))
        live.update(log_box(lines, f"MuscleMap log ({method}) - running {time.time() - t0:.0f} s"))
    proc.wait()
    live.update(log_box(lines, f"MuscleMap log ({method}) - {len(lines)} lines - {time.time() - t0:.0f} s"))

    stem = image_path.name.replace(".nii.gz", "")
    metrics_runs[method] = dict(csv=output_dir / f"{stem}_{method}_3component_results.csv",
                                fat=output_dir / f"{stem}_{method}_3component_fat_seg.nii.gz",
                                combined=output_dir / f"{stem}_{method}_3component_combined_seg.nii.gz",
                                softseg=output_dir / f"{stem}_gmm_fat_3component_softseg.nii.gz" if method == "gmm" else None)
    status_table([(method.upper(), proc.returncode == 0, f"exit code {proc.returncode}"),
                  ("Results", metrics_runs[method]["csv"].exists(), metrics_runs[method]["csv"].name),
                  ("Fat map", metrics_runs[method]["fat"].exists(), metrics_runs[method]["fat"].name)])

## 6 · Fat maps and fat percentage

The fat voxels in orange on top of the image, GMM in the top row and k-means in the bottom row. The slider sets the transparency. Then the GMM's posterior probability of fat as a heatmap, and finally the fat percentage and fat volume of the three muscles for both methods, read from the two CSV files.

In [ ]:
fat_maps = {m: nib.load(r["fat"]).get_fdata() > 0 for m, r in metrics_runs.items()}
FAT_RGB = [249, 115, 22]                              # MuscleMap orange

def fat_rgba(mask2d):
    rgba = np.zeros(mask2d.shape + (4,), dtype=np.uint8)
    rgba[mask2d] = FAT_RGB + [255]
    return rgba

methods = list(fat_maps)
method_names = {"gmm": "GMM", "kmeans": "k-means"}
fig = make_subplots(rows=len(methods), cols=n_slices, horizontal_spacing=0.01, vertical_spacing=0.08,
                    subplot_titles=[f"slice {k}" for k in range(n_slices)] + [""] * (n_slices * (len(methods) - 1)),
                    row_titles=[method_names[m] for m in methods])
overlay_idx = []
for r, m in enumerate(methods, start=1):
    for k in range(n_slices):
        fig.add_trace(go.Image(z=to_rgb(k), hoverinfo="skip"), row=r, col=k + 1)
        fig.add_trace(go.Image(z=fat_rgba(to_display(fat_maps[m][:, :, k])[crop]), colormodel="rgba", opacity=0.7,
                               hoverinfo="skip"), row=r, col=k + 1)
        overlay_idx.append(len(fig.data) - 1)
        fig.update_xaxes(visible=False, row=r, col=k + 1)
        fig.update_yaxes(visible=False, scaleanchor=f"x{(r - 1) * n_slices + k + 1 if (r, k) != (1, 0) else ''}", row=r, col=k + 1)

steps = [dict(method="restyle", label=f"{a:.1f}", args=[{"opacity": a}, overlay_idx]) for a in np.arange(0, 1.01, 0.1)]
for ann in fig.layout.annotations:                   # row titles: larger, upright, at the left
    if ann.text in method_names.values():
        ann.update(font=dict(size=18, color="#111827"), textangle=0, x=0.0, xanchor="right", xshift=-6)
fig.update_layout(height=int(40 + len(methods) * (panel_height(n_slices, fig_width=920) + 30) + 100),
                  margin=dict(l=90, r=10, t=40, b=90), paper_bgcolor=CLEAR, plot_bgcolor=CLEAR,
                  sliders=[dict(active=7, steps=steps, currentvalue=dict(prefix="fat opacity "), pad=dict(t=10), **SLIDER)])
fig.show()

### GMM posterior probability of fat

The GMM does not only assign each voxel to a component, it also gives the probability that a voxel belongs to the fat component. MuscleMap writes this as a soft segmentation (`…_gmm_fat_3component_softseg.nii.gz`). Shown here as a heatmap on the image: dark means almost certainly muscle, bright yellow means almost certainly fat, and the intermediate colours mark the voxels where the decision is uncertain.

In [ ]:
prob = nib.load(metrics_runs["gmm"]["softseg"]).get_fdata().astype(float)
prob[seg == 0] = np.nan                              # transparent outside the three muscles

fig = make_subplots(rows=1, cols=n_slices, horizontal_spacing=0.01, subplot_titles=[f"slice {k}" for k in range(n_slices)])
for k in range(n_slices):
    fig.add_trace(go.Image(z=to_rgb(k), hoverinfo="skip"), row=1, col=k + 1)
    fig.add_trace(go.Heatmap(z=to_display(prob[:, :, k])[crop], zmin=0, zmax=1, colorscale="Inferno", opacity=0.85,
                             showscale=(k == n_slices - 1), hoverongaps=False,
                             colorbar=dict(title="P(fat)", thickness=14, len=0.8, tickvals=[0, 0.25, 0.5, 0.75, 1]),
                             hovertemplate="P(fat) = %{z:.2f}<extra></extra>"), row=1, col=k + 1)
    fig.update_xaxes(visible=False, row=1, col=k + 1)
    fig.update_yaxes(visible=False, scaleanchor=f"x{k + 1 if k else ''}", row=1, col=k + 1)
fig.update_layout(height=int(36 + panel_height(n_slices, fig_width=940) + 30), margin=dict(l=10, r=70, t=36, b=10),
                  paper_bgcolor=CLEAR, plot_bgcolor=CLEAR)
fig.show()

In [ ]:
frames_ = []
for m, r in metrics_runs.items():
    df = pd.read_csv(r["csv"]).set_index("Label")
    sub = df.loc[muscles["label"].values, ["Fat (%)", "Fat volume (ml)", "Total volume (ml)"]]
    sub.index = muscles.index
    sub.columns = pd.MultiIndex.from_product([[m.upper() if m == "gmm" else "k-means"], ["Fat (%)", "Fat volume (ml)", "Total volume (ml)"]])
    frames_.append(sub)
fat_table = pd.concat(frames_, axis=1)
fat_table = fat_table.loc[:, [c for c in fat_table.columns if c[1] != "Total volume (ml)"] + [fat_table.columns[2]]]
fat_table.columns = [f"{m} · {q}" if q != "Total volume (ml)" else "Total volume (ml)" for m, q in fat_table.columns]

styled = (fat_table.style
          .format("{:.1f}")
          .bar(subset=[c for c in fat_table.columns if c.endswith("Fat (%)")], color="#f9731666", vmin=0, vmax=max(50, float(fat_table.filter(like="Fat (%)").max().max())))
          .apply(lambda col: [f"border-left: 10px solid {colors[muscles.loc[i, 'label']]}" for i in col.index], subset=[fat_table.columns[0]])
          .set_caption("Fat fraction and fat volume per muscle · 3-component GMM versus k-means")
          .set_table_styles([{"selector": "caption", "props": "caption-side: top; color: #6e7781; padding-bottom: 6px;"}]))
display(styled)

### Fat along the muscle, normalised to 0–100 %

The same layout as the CSA plot, now for fat: per slice, the fat voxels divided by all voxels of the muscle. Cranial is 0 %, caudal 100 %, cubic-spline interpolation to 101 points. Top row GMM, bottom row k-means; solid is left, dashed is right, dots are the measured slices.

In [ ]:
fat_pct_per_slice = {}
for m, fat in fat_maps.items():
    for v in present:
        lab = seg == v
        fat_pct_per_slice[(m, v)] = np.array([100 * (fat[:, :, k] & lab[:, :, k]).sum() / max(lab[:, :, k].sum(), 1)
                                              for k in range(n_slices)])[order]

fig = make_subplots(rows=len(fat_maps), cols=len(TARGETS), shared_yaxes=True, horizontal_spacing=0.04, vertical_spacing=0.14,
                    subplot_titles=[f"{structure} · {method_names[m]}" for m in fat_maps for structure in TARGETS])
for r, m in enumerate(fat_maps, start=1):
    for col, structure in enumerate(TARGETS, start=1):
        for side, dash in (("Left", "solid"), ("Right", "dash")):
            if (structure, side) not in muscles.index:
                continue
            v = muscles.loc[(structure, side), "label"]
            y = fat_pct_per_slice[(m, v)]
            c = colors[v]
            fig.add_trace(go.Scatter(x=length_pct, y=np.clip(resample(y, length_pct), 0, 100), mode="lines", name=side,
                                     legendgroup=side, showlegend=(r == 1 and col == 1), line=dict(color=c, width=3, dash=dash),
                                     hovertemplate="%{x:.0f} %<br>fat %{y:.1f} %<extra>" + f"{side} {structure} · {method_names[m]}</extra>"),
                          row=r, col=col)
            fig.add_trace(go.Scatter(x=x_slices, y=y, mode="markers", showlegend=False, legendgroup=side,
                                     marker=dict(color=c, size=9, line=dict(color="white", width=1)), hoverinfo="skip"),
                          row=r, col=col)
        fig.update_xaxes(title_text="Length (%)" if r == len(fat_maps) else "", range=[0, 100], dtick=20, gridcolor="#e5e7eb", row=r, col=col)
    fig.update_yaxes(title_text="Fat (%)", range=[0, 100], gridcolor="#e5e7eb", row=r, col=1)
fig.update_layout(height=720, paper_bgcolor=CLEAR, plot_bgcolor=CLEAR, margin=dict(l=10, r=10, t=50, b=10),
                  legend=dict(bgcolor="rgba(0,0,0,0)", orientation="h", y=-0.12, x=0.5, xanchor="center"))
fig.show()

## Where to go from here

* `mm_extract_metrics -m dixon` takes fat and water images from a Dixon acquisition and gives a true fat fraction; `-m average` works on fat-fraction maps and CT.
* `mm_register_to_template` brings segmentations into a common template space for group analyses.
* All labels, models and documentation: https://musclemap.github.io/MuscleMap/

<div style="border:1px solid #000000;border-left:5px solid #f97316;background:#ffffff;border-radius:0.5rem;padding:10px 16px;margin:10px 0;color:#111827;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;">Always look at the segmentation before you trust the numbers. MuscleMap recommends checking and, where needed, correcting segmentations in <a href="https://www.slicer.org/" style="color:#0550ae;">3D Slicer</a> or <a href="http://www.itksnap.org/" style="color:#0550ae;">ITK-SNAP</a>.</div>